In [ ]:
# Install required packages
%pip install xarray rioxarray netCDF4 numpy pandas fsspec requests s3fs aiohttp

In [ ]:
import os
from datetime import datetime
import requests

import pandas as pd 
import numpy as np

import fsspec
import xarray as xr
import s3fs

## USA Bounding Box Configuration

In [ ]:
# USA bounding box (includes Alaska and Hawaii)
USA_BBOX = {
    'lat_min': 18.91619,
    'lat_max': 71.3577635769,
    'lon_min': -171.791110603,
    'lon_max': -66.96466
}

print(f"USA Bounding Box:")
print(f"  Latitude: {USA_BBOX['lat_min']} to {USA_BBOX['lat_max']}")
print(f"  Longitude: {USA_BBOX['lon_min']} to {USA_BBOX['lon_max']}")

## Download Function

In [ ]:
def get_power_data(url, var, output_dir='USA_NASA_POWER'):
    """Download NASA POWER data for USA region"""
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Output filename
    output_file = os.path.join(output_dir, f"{var}.nc")
    
    # Check if already downloaded
    if os.path.exists(output_file):
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
        print(f"✓ Already exists: {var} ({file_size:.2f} MB)")
        return
    
    # Try direct HTTP access
    try:
        # First, check if the resource exists
        metadata_url = f'{url}/.zmetadata'
        response = requests.head(metadata_url)
        print(f"Resource check status: {response.status_code}")
        
        # Open the dataset
        ds = xr.open_dataset(url, engine='zarr')
        
        print(f"Dataset opened successfully! Processing variable: {var}")
        print(f"Dataset dimensions: {ds.dims}")
        
        # Select USA region and time range
        ds_region = ds[var].sel(
            lat=slice(USA_BBOX['lat_min'], USA_BBOX['lat_max']),
            lon=slice(USA_BBOX['lon_min'], USA_BBOX['lon_max']),
            time=slice("1990-01-01", "2024-12-31")
        ).load()
        
        print(f"Region data shape: {ds_region.shape}")
        
        # Save to NetCDF
        ds_region.to_netcdf(output_file)
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
        print(f"✓ Saved: {output_file} ({file_size:.2f} MB)\n")
        
    except Exception as e:
        print(f"Error with direct method: {e}")
        print("Attempting alternative approach...\n")
        
        try:
            # Try with fsspec without consolidated metadata
            mapper = fsspec.get_mapper(url)
            ds = xr.open_zarr(mapper, consolidated=False)
            
            print(f"Dataset opened with alternative method! Processing variable: {var}")
            
            ds_region = ds[var].sel(
                lat=slice(USA_BBOX['lat_min'], USA_BBOX['lat_max']),
                lon=slice(USA_BBOX['lon_min'], USA_BBOX['lon_max']),
                time=slice("1990-01-01", "2024-12-31")
            ).load()
            
            # Save to NetCDF
            ds_region.to_netcdf(output_file)
            file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
            print(f"✓ Saved: {output_file} ({file_size:.2f} MB)\n")
            
        except Exception as e2:
            print(f"✗ Alternative approach also failed: {e2}\n")

## Option 1: Download Subset of Variables

Common solar radiation and meteorological variables

In [ ]:
# Daily temporal dataset URL
url = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_daily_temporal_lst.zarr'

# Selected variables - most commonly used
selected_variables = [
    'ALLSKY_SFC_SW_DWN',      # All Sky Surface Shortwave Downward Irradiance
    'ALLSKY_SFC_LW_DWN',      # All Sky Surface Longwave Downward Irradiance
    'ALLSKY_SFC_PAR_TOT',     # All Sky Surface PAR Total
    'ALLSKY_SRF_ALB',         # All Sky Surface Albedo
    'T2M',                     # Temperature at 2 Meters
    'T2M_MAX',                 # Maximum Temperature at 2 Meters
    'T2M_MIN',                 # Minimum Temperature at 2 Meters
    'RH2M',                    # Relative Humidity at 2 Meters
    'PRECTOTCORR',            # Precipitation Corrected
    'WS2M',                    # Wind Speed at 2 Meters
    'PS',                      # Surface Pressure
]

print(f"Downloading {len(selected_variables)} variables for USA")
print(f"Date range: 1990-01-01 to 2024-12-31")
print(f"{'='*60}\n")

for i, var in enumerate(selected_variables, 1):
    print(f"[{i}/{len(selected_variables)}] Processing: {var}")
    get_power_data(url, var)

print(f"\n{'='*60}")
print("Download complete!")

## Option 2: Download All Available Variables

Downloads all variables from the NASA POWER dataset (47 variables)

In [ ]:
# All available variables in NASA POWER daily dataset
url = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_daily_temporal_lst.zarr'

all_variables = [
    'AIRMASS', 'ALLSKY_KT', 'ALLSKY_NKT', 'ALLSKY_SFC_LW_DWN', 'ALLSKY_SFC_LW_UP',
    'ALLSKY_SFC_PAR_DIFF', 'ALLSKY_SFC_PAR_DIRH', 'ALLSKY_SFC_PAR_TOT',
    'ALLSKY_SFC_SW_DIFF', 'ALLSKY_SFC_SW_DIRH', 'ALLSKY_SFC_SW_DNI',
    'ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_UP', 'ALLSKY_SFC_UV_INDEX',
    'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SRF_ALB', 'AOD_55',
    'AOD_55_ADJ', 'AOD_84', 'CLOUD_AMT', 'CLOUD_AMT_DAY', 'CLOUD_AMT_NIGHT',
    'CLOUD_OD', 'CLRSKY_DAYS', 'CLRSKY_KT', 'CLRSKY_NKT', 'CLRSKY_SFC_LW_DWN',
    'CLRSKY_SFC_LW_UP', 'CLRSKY_SFC_PAR_DIFF', 'CLRSKY_SFC_PAR_DIRH',
    'CLRSKY_SFC_PAR_TOT', 'CLRSKY_SFC_SW_DIFF', 'CLRSKY_SFC_SW_DIRH',
    'CLRSKY_SFC_SW_DNI', 'CLRSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_UP',
    'CLRSKY_SRF_ALB', 'MIDDAY_INSOL', 'ORIGINAL_ALLSKY_SFC_SW_DIFF',
    'ORIGINAL_ALLSKY_SFC_SW_DIRH', 'PSH', 'PW', 'SRF_ALB_ADJ',
    'TOA_SW_DNI', 'TOA_SW_DWN', 'TS_ADJ'
]

print(f"Downloading ALL {len(all_variables)} variables for USA")
print(f"Date range: 1990-01-01 to 2024-12-31")
print(f"{'='*60}\n")

for i, var in enumerate(all_variables, 1):
    print(f"[{i}/{len(all_variables)}] Processing: {var}")
    get_power_data(url, var)

print(f"\n{'='*60}")
print("All variables downloaded!")

## Option 3: Download Monthly Data

Downloads monthly aggregated data (smaller file sizes)

In [ ]:
# Monthly dataset URL
url_monthly = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_monthly_temporal_lst.zarr'

# Try to open and download all variables from monthly dataset
try:
    ds = xr.open_dataset(url_monthly, engine='zarr')
    print(f"Monthly dataset opened successfully!")
    print(f"Available variables: {list(ds.data_vars)}")
    print(f"\n{'='*60}\n")
    
    # Download all variables
    for i, var in enumerate(ds.data_vars, 1):
        print(f"[{i}/{len(ds.data_vars)}] Processing: {var}")
        get_power_data(url_monthly, var, output_dir='USA_NASA_POWER_Monthly')
    
    print(f"\n{'='*60}")
    print("Monthly data download complete!")
    
except Exception as e:
    print(f"Error accessing monthly dataset: {e}")